## Experiment Set 5: Dynamic Batching, max batch size 16

Dynamic batching, poisson arrivals, asynchronous calls, varying arrival rates

To edit the model configuration:

``` bash
# runs on node-serve-system
nano ~/serve-system-chi/models/food_classifier_onnx/config.pbtxt
```

Current config:

``` bash
name: "food_classifier_onnx"
backend: "onnxruntime"
max_batch_size: 16
input [
  {
    name: "input"  # has to match ONNX model's input name
    data_type: TYPE_FP32
    dims: [3, 224, 224]  # has to match ONNX input shape
  }
]
output [
  {
    name: "output"  # has to match ONNX model output name
    data_type: TYPE_FP32  # output is a list of probabilities
    dims: [11]  #
  }
]
  instance_group [
    {
      count: 1
      kind: KIND_GPU
      gpus: [ 0 ]
    }
]

dynamic_batching {
    preferred_batch_size: [16]
    max_queue_delay_microseconds: 100
}

```
Save the file (use Ctrl+O then Enter, then Ctrl+X).

Re-build the container image with this change:

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml build triton_server
```

and then bring the server back up:

``` bash
# runs on node-serve-system
docker compose -f ~/serve-system-chi/docker/docker-compose-triton.yaml up triton_server --force-recreate -d
```

and use

``` bash
# runs on node-serve-system
docker logs triton_server
```

to make sure the server comes up and is ready.

Before we benchmark this service again, let’s get some pre-benchmark stats about how many requests have been served, broken down by batch size. (If you’ve just restarted the server, it would be zero!)

In [1]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 20:100:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_16_results/batch_16_20-100.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 100 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 20 inference requests per second
Failed to obtain stable measurement.
Failed to obtain stable measurement.



In [2]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_16_results/stats_20-100.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778460226086,
            "inference_count": 1157,
            "execution_count": 384,
            "inference_stats": {
                "success": {
                    "count": 1157,
                    "ns": 6615810941293
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 1157,
                    "ns": 6227766698317
                },
                "compute_input": {
                    "count": 1157,
                    "ns": 3782427502
                },
                "compute_infer": {
                    "count": 1157,
                    "ns": 384082801537
                },
                "compute_output": {
                    "count": 1157,
                    "ns": 49052276
                },
                

In [3]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 120:200:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_16_results/batch_16_120-200.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 200 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 120 inference requests per second
Failed to retrieve results from inference request.
Thread [0] had error: Deadline Exceeded

Thread [1] had error: Deadline Exceeded

Thread [2] had error: Deadline Exceeded

Thread [3] had error: Deadline Exceeded




In [4]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_16_results/stats_120-200.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778460432016,
            "inference_count": 23439,
            "execution_count": 16174,
            "inference_stats": {
                "success": {
                    "count": 23439,
                    "ns": 36775890303003
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 23439,
                    "ns": 35836145676980
                },
                "compute_input": {
                    "count": 23439,
                    "ns": 28500392221
                },
                "compute_infer": {
                    "count": 23439,
                    "ns": 909004909457
                },
                "compute_output": {
                    "count": 23439,
                    "ns": 631937795
                },
    

In [5]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 220:300:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_16_results/batch_16_220-300.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 300 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 220 inference requests per second
  Client: 
    Request count: 4261
    Throughput: 226.039 infer/sec
    Avg latency: 8196 usec (standard deviation 3514 usec)
    p50 latency: 7353 usec
    p90 latency: 11970 usec
    p95 latency: 13615 usec
    p99 latency: 18775 usec
    Avg HTTP time: 8101 usec (send/recv 404 usec + response wait 7697 usec)
  Server: 
    Inference count: 4262
    Execution count: 3619
    Successful request count: 4262
    Avg request latency: 5192 usec (overhead 25 usec + queue 1788 usec + compute input 161 usec + compute infer 3204 usec + compute output 13 usec)
Request Rate: 240 inference requests per se

In [6]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_16_results/stats_220-300.json

{
    "model_stats": [
        {
            "name": "food_classifier_onnx",
            "version": "1",
            "last_inference": 1778460571228,
            "inference_count": 57908,
            "execution_count": 44452,
            "inference_stats": {
                "success": {
                    "count": 57908,
                    "ns": 36958326124511
                },
                "fail": {
                    "count": 0,
                    "ns": 0
                },
                "queue": {
                    "count": 57908,
                    "ns": 35902521722150
                },
                "compute_input": {
                    "count": 57908,
                    "ns": 34629700270
                },
                "compute_infer": {
                    "count": 57908,
                    "ns": 1017634912087
                },
                "compute_output": {
                    "count": 57908,
                    "ns": 1086817794
                },
  

In [ ]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 320:400:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_16_results/batch_16_320-400.txt

*** Measurement Settings ***
  Batch size: 1
  Service Kind: TRITON
  Using "time_windows" mode for stabilization
  Stabilizing using average throughput
  Measurement window: 5000 msec
  Latency limit: 0 msec
  Request Rate limit: 400 requests per seconds
  Using poisson distribution on request generation
  Using asynchronous calls for inference

Request Rate: 320 inference requests per second


In [ ]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_16_results/stats_320-400.json

In [ ]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 420:500:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_16_results/batch_16_420-500.txt

In [ ]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_16_results/stats_420-500.json

In [ ]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 520:600:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_16_results/batch_16_520-600.txt

In [ ]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_16_results/stats_520-600.json

In [ ]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 620:700:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_16_results/batch_16_620-700.txt 

In [ ]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_16_results/stats_620-700.json

In [ ]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 720:800:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_16_results/batch_16_720-800.txt 

In [ ]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_16_results/stats_720-800.json

In [ ]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 820:900:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_16_results/batch_16_820-900.txt 

In [ ]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_16_results/stats_820-900.json

In [ ]:
# runs inside the Jupyter container on node-serve-system
perf_analyzer -u triton_server:8000 -m food_classifier_onnx -b 1 --shape IMAGE:3,224,224 --request-rate-range 920:1000:20 --async --request-distribution poisson | tee /home/jovyan/work/batch_16_results/batch_16_920-1000.txt 

In [ ]:
curl -s http://triton_server:8000/v2/models/food_classifier_onnx/versions/1/stats | python -m json.tool | tee /home/jovyan/work/batch_16_results/stats_920-1000.json